<a href="https://colab.research.google.com/github/prasath25/Hands-on/blob/main/Experiment7_StudentOrchestrator_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment 7 — Agent Orchestration with LangGraph (Colab version)
### Theme: "Student Query Router"

The **Google Colab version** of Experiment 7 — avoids local pip/venv/Python-version issues entirely by running on Colab's fully-supported Python, with internet already available for the one-time (per session) model download.

**Still no API key of any kind.** The model (`Qwen/Qwen2.5-1.5B-Instruct`) is a public HuggingFace model — no login/token needed.

Unlike Experiment 6's ReAct agent (which re-decides its whole plan at every step), this graph classifies the question **once** and routes deterministically to a handler — with a bounded retry loop (`clarify`) and a circuit breaker so it can never loop forever.

### Before you run
`Runtime → Change runtime type → T4 GPU` recommended (CPU also works, just slower).
Run the cells **top to bottom, in order**.


## 0. Install dependencies (one-time per session)

In [1]:
!pip install -q -U langgraph transformers accelerate huggingface_hub

import torch
print("GPU available:", torch.cuda.is_available())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 13.4 MB/s eta 0:00:00
GPU available: True


---
## 1. Tools — same pure-Python logic as Experiment 6

`handle_gpa` and `handle_study_plan` will call these; `handle_faq` needs no LLM call at all.


In [2]:
import re, json as _json

# ---- GPA Calculator ----
GRADE_POINTS = {
    "A+": 4.0, "A": 4.0, "A-": 3.7,
    "B+": 3.3, "B": 3.0, "B-": 2.7,
    "C+": 2.3, "C": 2.0, "C-": 1.7,
    "D+": 1.3, "D": 1.0, "D-": 0.7,
    "F": 0.0,
}

def gpa_calculator(payload: str) -> str:
    try:
        data = _json.loads(payload)
    except _json.JSONDecodeError as exc:
        return f'Error: input must be valid JSON like {{"grades": ["A","B+"], "credits": [3,4]}}. Parse error: {exc}'
    grades = data.get("grades")
    credits = data.get("credits")
    if not isinstance(grades, list) or not isinstance(credits, list):
        return 'Error: JSON must contain a "grades" list and a "credits" list.'
    if len(grades) != len(credits):
        return "Error: 'grades' and 'credits' lists must be the same length."
    if not grades:
        return "Error: no grades provided."
    total_points, total_credits, unknown = 0.0, 0.0, []
    for grade, credit in zip(grades, credits):
        key = str(grade).strip().upper()
        if key not in GRADE_POINTS:
            unknown.append(grade)
            continue
        try:
            c = float(credit)
        except (TypeError, ValueError):
            return f"Error: credit value {credit!r} is not a number."
        total_points += GRADE_POINTS[key] * c
        total_credits += c
    if unknown:
        return f"Error: unrecognized grade(s) {unknown}. Use standard letter grades (A, A-, B+, ... F)."
    if total_credits == 0:
        return "Error: total credits cannot be zero."
    gpa = total_points / total_credits
    return f"GPA = {gpa:.2f} (on a 4.0 scale), based on {total_credits:.0f} total credit hours across {len(grades)} course(s)."


# ---- Study Planner ----
def study_planner(payload: str) -> str:
    try:
        data = _json.loads(payload)
    except _json.JSONDecodeError as exc:
        return f"Error: input must be valid JSON. Parse error: {exc}"
    hours_per_day = data.get("hours_per_day")
    subjects = data.get("subjects")
    if not isinstance(subjects, list) or not subjects:
        return 'Error: JSON must include a non-empty "subjects" list.'
    try:
        hours_per_day = float(hours_per_day)
    except (TypeError, ValueError):
        return 'Error: "hours_per_day" must be a number.'
    if hours_per_day <= 0:
        return 'Error: "hours_per_day" must be greater than zero.'
    scores = []
    for subj in subjects:
        name = subj.get("name")
        difficulty = subj.get("difficulty")
        days_left = subj.get("days_left")
        if not name:
            return "Error: every subject needs a 'name'."
        try:
            difficulty = float(difficulty)
            days_left = float(days_left)
        except (TypeError, ValueError):
            return f"Error: subject '{name}' needs numeric 'difficulty' and 'days_left'."
        if days_left <= 0:
            return f"Error: 'days_left' for '{name}' must be greater than zero."
        scores.append((name, difficulty / days_left, days_left))
    total_urgency = sum(s[1] for s in scores)
    if total_urgency == 0:
        return "Error: all urgency scores computed to zero, check difficulty values."
    lines = [f"Suggested daily study plan ({hours_per_day:.1f} hours/day total):"]
    for name, urgency, days_left in sorted(scores, key=lambda s: -s[1]):
        allocated = hours_per_day * (urgency / total_urgency)
        lines.append(f"  - {name}: {allocated:.1f} hrs/day (exam in {days_left:.0f} day(s))")
    return "\n".join(lines)


# ---- Course FAQ Lookup (local, keyword-based, no LLM call) ----
STUDENT_FAQ = [
    {"question": "What GPA do I need to stay in good academic standing?",
     "answer": "Most programs require a cumulative GPA of at least 2.0 on a 4.0 scale to remain in good academic standing."},
    {"question": "How many credits are typically needed to graduate?",
     "answer": "A standard undergraduate degree usually requires 120-130 credit hours; many master's programs require 30-36."},
    {"question": "What counts as plagiarism?",
     "answer": "Plagiarism includes copying text, code, or ideas from another source without proper citation, or submitting someone else's work as your own."},
    {"question": "How do I request a deadline extension?",
     "answer": "Contact your instructor directly, before the deadline if possible, explaining the reason and proposing a new date."},
    {"question": "What is the recommended way to prepare for final exams?",
     "answer": "Start reviewing 1-2 weeks before the exam, break topics into daily study blocks, and practice with past papers."},
    {"question": "How is a weighted GPA different from an unweighted GPA?",
     "answer": "A weighted GPA gives extra points for harder courses, often on a 5.0 scale; an unweighted GPA treats all courses equally on a 4.0 scale."},
    {"question": "What should I do if I am struggling with a course?",
     "answer": "Reach out to your instructor during office hours, form a study group, and use tutoring center resources."},
    {"question": "How many hours should I study per credit hour each week?",
     "answer": "A common guideline is 2-3 hours of independent study per week for every 1 credit hour of class time."},
]

_STOPWORDS = {"a","an","the","is","are","am","do","does","did","i","you","what","how","when","where",
              "why","should","for","of","to","my","me","in","on","and","or","it","this","that","can",
              "will","would","get","need","needed","ask"}
_WORD_RE = re.compile(r"[a-zA-Z]+")

def _tokenize(text):
    return set(w.lower() for w in _WORD_RE.findall(text)) - _STOPWORDS

def course_faq_lookup(query: str) -> str:
    query = query.strip()
    if not query:
        return "Error: empty query."
    query_words = _tokenize(query)
    if not query_words:
        return "No relevant FAQ found for that query."
    best_score, best_entry = 0, None
    for entry in STUDENT_FAQ:
        overlap = len(query_words & _tokenize(entry["question"]))
        if overlap > best_score:
            best_score, best_entry = overlap, entry
    if best_entry is None or best_score == 0:
        return "No relevant FAQ found for that query. Try rephrasing with different keywords."
    return f"FAQ: {best_entry['question']}\nAnswer: {best_entry['answer']}"

print("Tools ready.")


Tools ready.


---
## 2. Load the local model (public, no API key, no login)

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto", device_map="auto")

text_gen_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150,
    do_sample=False,
    return_full_text=False,
)

def llm(prompt: str) -> str:
    return text_gen_pipeline(prompt)[0]["generated_text"]

print("Model loaded.")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Model loaded.


---
## 3. Graph state and node functions

Same logic as `graph/state.py` and `graph/nodes.py` in the VS Code project.


In [6]:
# The functions and classes previously defined here have been moved to cell qwkvtEeSAXpq
# to ensure they are in scope when the graph is built.
print("Node definitions handled in graph building cell.")

Node definitions handled in graph building cell.


---
## 4. Build the graph

Same wiring as `graph/build_graph.py` — `classify_intent` fans out to 4 handlers, with a bounded `clarify` retry loop and a circuit breaker.


In [15]:
from langgraph.graph import StateGraph, START, END
from typing import Optional, TypedDict
import re, json as _json

class AgentState(TypedDict, total=False):
    question: str
    intent: Optional[str]
    retries: int
    answer: str

VALID_INTENTS = {"GPA", "STUDY_PLAN", "FAQ", "GENERAL"}

def _extract_json(text: str):
    # Try to find JSON in a markdown code block first (```json\n...\n``` or ```\n...\n```)
    json_block_match = re.search(r'```(?:json)?\\n(.*?)\\n```', text, re.DOTALL)
    if json_block_match:
        json_str = json_block_match.group(1).strip()
        try:
            return _json.loads(json_str)
        except _json.JSONDecodeError as e:
            print(f"DEBUG: _extract_json: JSON parse error in markdown block: {e}, attempting to parse: {json_str!r}")
            # Fallback to general JSON extraction if markdown block is malformed or not found

    # Fallback to finding any standalone JSON object
    # Use non-greedy match for content to avoid issues with extra text outside or nested structures
    match = re.search(r'\\{.*?\\}', text, re.DOTALL)
    if match:
        json_str = match.group(0)
        try:
            return _json.loads(json_str)
        except _json.JSONDecodeError as e:
            print(f"DEBUG: _extract_json: JSON parse error in general match: {e}, attempting to parse: {json_str!r}")
            # Consider more advanced cleanup here if needed, but for now, just pass.
    return None


def classify_intent(state, llm):
    question = state["question"]
    retries = state.get("retries", 0)
    examples = ""
    if retries > 0:
        examples = (
            "\\nExamples:\\n"
            '"What GPA do I have with an A and a B?" -> GPA\\n'
            '"How should I split 3 hours between two classes?" -> STUDY_PLAN\\n'
            '"How do I request an extension?" -> FAQ\\n'
            '"What time is it?" -> GENERAL\\n'
        )
    prompt = (
        "Classify the student's question into exactly ONE of these words: "
        "GPA, STUDY_PLAN, FAQ, GENERAL. Reply with only that one word.\\n"
        f"{examples}\\n"
        f'Question: "{question}"\\n'
        "Category:"
    )
    response = llm(prompt).strip().upper()
    print(f"[classify_intent] raw model response: {response!r}")

    found = None
    # Attempt to extract the intent by taking the first word from the first non-empty line
    lines = [line.strip() for line in response.split('\\n') if line.strip()]
    if lines:
        first_line = lines[0]
        # Use regex to find the first sequence of uppercase letters/underscore (valid intent format)
        word_match = re.match(r'\\b([A-Z_]+)\\b', first_line)
        if word_match:
            candidate_intent = word_match.group(1)
            if candidate_intent in VALID_INTENTS:
                found = candidate_intent

    # Fallback: if the above didn't yield a valid intent, try the original 'in response' check
    # This is less precise but provides a backup.
    if found is None:
        for candidate in VALID_INTENTS:
            if candidate in response:
                found = candidate
                break # Take the first one found in the response

    print(f"[classify_intent] resolved intent: {found}")
    return {"intent": found}


def clarify(state):
    retries = state.get("retries", 0) + 1
    print(f"[clarify] intent was unclear, retry #{retries} -- reclassifying with extra examples")
    return {"retries": retries}


def handle_gpa(state, llm):
    question = state["question"]
    prompt = (
        "Extract the letter grades and credit hours mentioned in this student "
        "question, and output ONLY a JSON object, nothing else.\\n"
        'Format: {"grades": ["A", "B+"], "credits": [3, 4]}\\n\\n'
        f'Question: "{question}"\\n\\nJSON:'
    )
    payload = _extract_json(llm(prompt))
    if payload is None:
        return {"answer": "I couldn't tell which grades and credit hours you meant. Try: \"What is my GPA with an A in a 3-credit class and a B+ in a 4-credit class?\""}
    return {"answer": gpa_calculator(_json.dumps(payload))}


def handle_study_plan(state, llm):
    question = state["question"]
    prompt = (
        "Extract the study plan details from this student question, and output "
        "ONLY a JSON object, nothing else.\\n"
        'Format: {"hours_per_day": 4, "subjects": [{"name": "Math", "difficulty": 4, "days_left": 3}]}}\\n\\n'
        f'Question: "{question}"\\n\\nJSON:'
    )
    payload = _extract_json(llm(prompt))
    if payload is None:
        return {"answer": "I couldn't tell your subjects, difficulty levels, days left, or hours available. Try: \"I have 4 hours a day, Math is difficulty 4 with 3 days left, History is difficulty 2 with 10 days left.\""}
    return {"answer": study_planner(_json.dumps(payload))}


def handle_faq(state):
    return {"answer": course_faq_lookup(state["question"])}


def handle_general(state, llm):
    question = state["question"]
    prompt = f"Answer this student's question helpfully and concisely.\\nQuestion: {question}\\nAnswer:"
    return {"answer": llm(prompt).strip()}

def build_graph(llm):
    builder = StateGraph(AgentState)

    builder.add_node("classify_intent", lambda s: classify_intent(s, llm))
    builder.add_node("clarify", clarify)
    builder.add_node("handle_gpa", lambda s: handle_gpa(s, llm))
    builder.add_node("handle_study_plan", lambda s: handle_study_plan(s, llm))
    builder.add_node("handle_faq", handle_faq)
    builder.add_node("handle_general", lambda s: handle_general(s, llm))

    builder.add_edge(START, "classify_intent")

    def route_from_classify(state):
        intent = state.get("intent")
        retries = state.get("retries", 0)
        if intent == "GPA":
            return "handle_gpa"
        if intent == "STUDY_PLAN":
            return "handle_study_plan"
        if intent == "FAQ":
            return "handle_faq"
        if intent == "GENERAL":
            return "handle_general"
        if retries >= 2:
            return "handle_general"  # circuit breaker -- never loop forever
        return "clarify"

    builder.add_conditional_edges(
        "classify_intent",
        route_from_classify,
        {
            "handle_gpa": "handle_gpa",
            "handle_study_plan": "handle_study_plan",
            "handle_faq": "handle_faq",
            "handle_general": "handle_general",
            "clarify": "clarify",
        },
    )
    builder.add_edge("clarify", "classify_intent")
    for node_name in ("handle_gpa", "handle_study_plan", "handle_faq", "handle_general"):
        builder.add_edge(node_name, END)

    return builder.compile()

graph = build_graph(llm)
print("Graph compiled.")

Graph compiled.


---
## 5. Try it out

Watch the `[classify_intent]` / `[clarify]` print lines to see which nodes the graph visits.


In [8]:
result = graph.invoke({"question": "What is my GPA with an A in a 3-credit class and a B+ in a 4-credit class?", "retries": 0})
print("\nFinal answer:", result["answer"])

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the docum

[classify_intent] raw model response: "GPA\n\nSTUDENT'S QUESTION ASKS ABOUT THEIR GPA, SO IT SHOULD BE CLASSIFIED AS 'GPA'. \n\nANSWER: GPA"
[classify_intent] resolved intent: GPA

Final answer: I couldn't tell which grades and credit hours you meant. Try: "What is my GPA with an A in a 3-credit class and a B+ in a 4-credit class?"


In [9]:
result = graph.invoke({"question": "I have 4 hours a day. Math is difficulty 4 with 3 days left, History is difficulty 2 with 10 days left.", "retries": 0})
print("\nFinal answer:", result["answer"])

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[classify_intent] raw model response: "STUDY_PLAN\n\nSTUDENT'S QUESTION ASKS ABOUT CREATING A STUDY PLAN BASED ON THEIR AVAILABLE TIME AND SUBJECT DIFFICULTIES. THEREFORE, IT SHOULD BE CLASSIFIED AS 'STUDY_PLAN'. \n\nREVISED CATEGORY: STUDY_PLAN\nSTUDYING FOR EXAMS CAN HELP YOU IMPROVE YOUR GRADES IN SCHOOL. GOOD LUCK! 🎓🎓💪 #STUDYSUCCESS #GPABOOST #EDUCATIONALGOALS #ACADEMICACHIEVEMENT #LEARNINGJOURNEY #STUDYPLANTIPS #TIMEMANAGEMENT #EXAMPREPARATION #EDUCATIONRESOURCES #SCHOOLLIFE #STUDYHABITS #STUDYSKILLS #STUDYSTRATEGY #STUDYROUTINE #STUDYTIPS #STUDYMOTIVATION #STUDYGOALS #STUDYSUPPORT #STUDYHELP #STUDYGUIDES #STUDYWORKBOOK #STUDYNOTES #STUDYMATERIALS #STUDYSESSION #"
[classify_intent] resolved intent: GPA

Final answer: I couldn't tell which grades and credit hours you meant. Try: "What is my GPA with an A in a 3-credit class and a B+ in a 4-credit class?"


In [10]:
result = graph.invoke({"question": "How do I request a deadline extension?", "retries": 0})
print("\nFinal answer:", result["answer"])

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[classify_intent] raw model response: "STUDY_PLAN\n\nSTUDENT'S QUESTION IS ASKING ABOUT REQUESTING AN EXTENSION ON A DEADLINE, WHICH FALLS UNDER THE CATEGORY OF **STUDY_PLAN**. THE QUESTION PERTAINS TO ACADEMIC OR STUDY-RELATED DEADLINES AND EXTENSIONS, SO IT FITS BEST IN THIS CATEGORY."
[classify_intent] resolved intent: STUDY_PLAN

Final answer: I couldn't tell your subjects, difficulty levels, days left, or hours available. Try: "I have 4 hours a day, Math is difficulty 4 with 3 days left, History is difficulty 2 with 10 days left."


In [11]:
result = graph.invoke({"question": "What is my GPA with an A in a 3-credit class and a B+ in a 4-credit class?", "retries": 0})
print("\nFinal answer:", result["answer"])


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[classify_intent] raw model response: "GPA\n\nSTUDENT'S QUESTION ASKS ABOUT THEIR GPA, SO IT SHOULD BE CLASSIFIED AS 'GPA'. \n\nANSWER: GPA"
[classify_intent] resolved intent: GPA

Final answer: I couldn't tell which grades and credit hours you meant. Try: "What is my GPA with an A in a 3-credit class and a B+ in a 4-credit class?"


In [12]:
result = graph.invoke({"question": "I have 4 hours a day. Math is difficulty 4 with 3 days left, History is difficulty 2 with 10 days left.", "retries": 0})
print("\nFinal answer:", result["answer"])


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[classify_intent] raw model response: "STUDY_PLAN\n\nSTUDENT'S QUESTION ASKS ABOUT CREATING A STUDY PLAN BASED ON THEIR AVAILABLE TIME AND SUBJECT DIFFICULTIES. THEREFORE, IT SHOULD BE CLASSIFIED AS 'STUDY_PLAN'. \n\nREVISED CATEGORY: STUDY_PLAN\nSTUDYING FOR EXAMS CAN HELP YOU IMPROVE YOUR GRADES IN SCHOOL. GOOD LUCK! 🎓🎓💪 #STUDYSUCCESS #GPABOOST #EDUCATIONALGOALS #ACADEMICACHIEVEMENT #LEARNINGJOURNEY #STUDYPLANTIPS #TIMEMANAGEMENT #EXAMPREPARATION #EDUCATIONRESOURCES #SCHOOLLIFE #STUDYHABITS #STUDYSKILLS #STUDYSTRATEGY #STUDYROUTINE #STUDYTIPS #STUDYMOTIVATION #STUDYGOALS #STUDYSUPPORT #STUDYHELP #STUDYGUIDES #STUDYWORKBOOK #STUDYNOTES #STUDYMATERIALS #STUDYSESSION #"
[classify_intent] resolved intent: GPA

Final answer: I couldn't tell which grades and credit hours you meant. Try: "What is my GPA with an A in a 3-credit class and a B+ in a 4-credit class?"


In [13]:
result = graph.invoke({"question": "How do I request a deadline extension?", "retries": 0})
print("\nFinal answer:", result["answer"])


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[classify_intent] raw model response: "STUDY_PLAN\n\nSTUDENT'S QUESTION IS ASKING ABOUT REQUESTING AN EXTENSION ON A DEADLINE, WHICH FALLS UNDER THE CATEGORY OF **STUDY_PLAN**. THE QUESTION PERTAINS TO ACADEMIC OR STUDY-RELATED DEADLINES AND EXTENSIONS, SO IT FITS BEST IN THIS CATEGORY."
[classify_intent] resolved intent: STUDY_PLAN

Final answer: I couldn't tell your subjects, difficulty levels, days left, or hours available. Try: "I have 4 hours a day, Math is difficulty 4 with 3 days left, History is difficulty 2 with 10 days left."


---
## 6. Ask your own question

Edit the string below and re-run this cell as many times as you like.


In [14]:
your_question = "What day is it today?"  # <-- edit this line

result = graph.invoke({"question": your_question, "retries": 0})
print("\nFinal answer:", result["answer"])


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[classify_intent] raw model response: "FAQ\nEXPLANATION: THE GIVEN QUESTION DOES NOT PERTAIN TO A SPECIFIC ACADEMIC PERFORMANCE METRIC (GPA), STUDY PLAN, OR GENERAL INFORMATION ABOUT STUDENTS' LIVES BEYOND THEIR STUDIES. IT ASKS FOR THE CURRENT DATE, WHICH FALLS UNDER THE CATEGORY OF 'FAQ' AS IT SEEKS FACTUAL INFORMATION RATHER THAN EDUCATIONAL GUIDANCE ON STUDYING OR ACADEMIC ACHIEVEMENTS."
[classify_intent] resolved intent: FAQ

Final answer: No relevant FAQ found for that query. Try rephrasing with different keywords.


---
## Troubleshooting

| Symptom | Fix |
|---|---|
| `pip install` cell errors | `Runtime → Restart session`, then re-run the install cell once more before anything else |
| Classification keeps landing on the wrong bucket | Expected occasionally with a 1.5B model — the `clarify` retry loop exists for this; try rephrasing with a clearer keyword ("GPA", "study plan", "extension") |
| `CUDA out of memory` | `Runtime → Change runtime type → CPU`, then re-run from the top |
| Session disconnected | Colab free-tier sessions can time out — just re-run all cells from the top |
